# Análise e limpeza do INCT

In [1]:
!pip install pandas


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
from pathlib import Path

import pandas as pd

# Pasta raiz do repositório (funciona rodando de notebooks/ ou da raiz)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ORIGINAIS = RAIZ / "dados" / "originais"

with open(ORIGINAIS / "INCT" / "data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
with pd.option_context("display.max_rows", None):
    display(df) 


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population
0,3.0,Aberta dos Morros,0,4696,0,9854,0
1,3.0,Agronomia,0,1771,0,2677,0
2,3.0,Anchieta,28,589,5,791,40
3,3.0,Arquipélago,4685,4688,100,6411,6411
4,3.0,Auxiliadora,0,1995,0,8909,0
5,3.0,Azenha,0,3317,0,12064,0
6,3.0,Bela Vista,0,1636,0,11819,0
7,3.0,Belém Novo,95,6012,2,9851,198
8,3.0,Belém Velho,0,5887,0,10893,0
9,3.0,Boa Vista,0,1875,0,9254,0


## Dica 1 — Prédios afetados devem crescer (ou ficar iguais) quando o rio sobe

**Regra esperada:** para um mesmo bairro, se o nível do Guaíba sobe, a água cobre tudo o que já cobria **e mais**.
Logo, `affected_buildings` deve ser **monotônico não-decrescente** em relação a `nivel_m`.

Qualquer caso em que o valor **diminui** de um nível para o seguinte é uma inconsistência.

In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ordena por bairro e nível (não confiar na ordem do arquivo — ver dica 3)
df = df.sort_values(["bairro", "nivel_m"]).reset_index(drop=True)
# Diferença em relação ao nível anterior do mesmo bairro
df["anterior"] = df.groupby("bairro")["affected_buildings"].shift(1)
df["diff"] = df["affected_buildings"] - df["anterior"]

quedas = df[df["diff"] < 0][["bairro", "nivel_m", "anterior", "affected_buildings", "diff"]]
print(f"{len(quedas)} quedas encontradas em {quedas['bairro'].nunique()} bairros")
quedas

12 quedas encontradas em 5 bairros


,bairro,nivel_m,anterior,affected_buildings,diff
214,Arquipélago,3.1,4685.0,4680,-5.0
219,Arquipélago,3.6,4685.0,4680,-5.0
224,Arquipélago,4.1,4685.0,4674,-11.0
229,Arquipélago,4.6,4685.0,4680,-5.0
234,Arquipélago,5.1,4685.0,4680,-5.0
237,Arquipélago,5.4,4688.0,4680,-8.0
1274,Centro Histórico,9.7,2050.0,0,-2050.0
1779,Espírito Santo,3.4,637.0,633,-4.0
1780,Espírito Santo,3.5,633.0,25,-608.0
1793,Espírito Santo,4.8,2101.0,47,-2054.0


In [4]:
dc = df.copy()
dc["affected_buildings_original"] = dc["affected_buildings"]
dc["tratamento_d1"] = ""
g = dc.groupby("bairro")["affected_buildings"]
ab = dc["affected_buildings"].astype(float)
viz_ant, viz_pos = g.shift(1), g.shift(-1)

# (a) Zeros isolados: 0 depois de o bairro já ter tido prédios afetados
ja_afetado = g.cummax().groupby(dc["bairro"]).shift(1) > 0
m_zero = (dc["affected_buildings"] == 0) & ja_afetado

# (b) Picos isolados: valor muito maior que os dois vizinhos
m_pico = dc["affected_buildings"] > 3 * np.maximum(viz_ant, viz_pos)

# (c) Patamar inicial alto: todos os níveis antes de uma queda > 50%
#     (desconsiderando as quedas que só existem por causa de um pico já marcado)
queda_grande = (dc["affected_buildings"] < 0.5 * viz_ant) & ~m_pico.groupby(dc["bairro"]).shift(1, fill_value=False) & ~m_zero
depois_da_queda = queda_grande.astype(int).groupby(dc["bairro"]).cumsum() > 0
tem_queda = queda_grande.groupby(dc["bairro"]).transform("any")
m_patamar = tem_queda & ~depois_da_queda

# (d) Ruído pequeno: bairros que AINDA têm quedas depois de remover (a)-(c);
#     neles, valores que se desviam do valor predominante (moda) do bairro
#     em até 1% do total de prédios
restante = dc["affected_buildings"].where(~(m_zero | m_pico | m_patamar))
cai_restante = restante.groupby(dc["bairro"]).transform(lambda s: (s.dropna().diff() < 0).any())
ainda_cai = cai_restante.astype(bool)
moda = g.transform(lambda s: s.mode().iloc[0])
m_ruido = (ainda_cai & (dc["affected_buildings"] != moda)
           & ((dc["affected_buildings"] - moda).abs() <= 0.01 * dc["total_buildings"])
           & ~(m_zero | m_pico | m_patamar))

for mask, rotulo in [(m_zero, "interpolado (zero isolado)"),
                     (m_pico, "interpolado (pico isolado)"),
                     (m_patamar, "interpolado (patamar inicial inconsistente)"),
                     (m_ruido, "interpolado (ruído)")]:
    dc.loc[mask, "tratamento_d1"] = rotulo
    ab[mask] = np.nan

# Interpolação linear pelo nível, dentro de cada bairro.
# limit_direction="both" preenche também o início da série (repete o primeiro valor válido).
ab = ab.groupby(dc["bairro"]).transform(
    lambda s: s.set_axis(dc.loc[s.index, "nivel_m"])
               .interpolate(method="index", limit_direction="both").values)
dc["affected_buildings"] = ab.round().astype(int)

# Recalcula os campos derivados com a mesma regra do INCT (a dica 2 discute essa regra)
dc["percentage"] = np.ceil(dc["affected_buildings"] / dc["total_buildings"] * 100).astype(int)
dc["affected_population"] = np.ceil(dc["population"] * dc["percentage"] / 100).astype(int)

alteradas = dc[dc["tratamento_d1"] != ""]
print(alteradas["tratamento_d1"].value_counts())
alteradas[["bairro", "nivel_m", "affected_buildings_original", "affected_buildings", "tratamento_d1"]]

tratamento_d1
interpolado (ruído)                            7
interpolado (patamar inicial inconsistente)    5
interpolado (zero isolado)                     4
interpolado (pico isolado)                     1
Name: count, dtype: int64


,bairro,nivel_m,affected_buildings_original,affected_buildings,tratamento_d1
213,Arquipélago,3.0,4685,4680,interpolado (ruído)
218,Arquipélago,3.5,4685,4680,interpolado (ruído)
223,Arquipélago,4.0,4685,4680,interpolado (ruído)
224,Arquipélago,4.1,4674,4680,interpolado (ruído)
228,Arquipélago,4.5,4685,4680,interpolado (ruído)
233,Arquipélago,5.0,4685,4680,interpolado (ruído)
236,Arquipélago,5.3,4688,4680,interpolado (ruído)
1274,Centro Histórico,9.7,0,2077,interpolado (zero isolado)
1275,Centro Histórico,9.8,0,2103,interpolado (zero isolado)
1775,Espírito Santo,3.0,637,25,interpolado (patamar inicial inconsistente)


### Verificação: não pode sobrar nenhuma queda

In [5]:
bairros_prob = sorted(quedas["bairro"].unique())
d2 = dc.groupby("bairro")["affected_buildings"].diff()
print("Quedas restantes:", int((d2 < 0).sum()))

fig = px.line(dc[dc["bairro"].isin(bairros_prob)], x="nivel_m", y="affected_buildings",
              facet_col="bairro", facet_col_wrap=3, markers=True,
              category_orders={"bairro": sorted(bairros_prob)},
              labels={"nivel_m": "Nível (m)", "affected_buildings": "Prédios afetados"},
              title="Depois do tratamento (cinza tracejado = original)", height=650)
orig = px.line(df[df["bairro"].isin(bairros_prob)], x="nivel_m", y="affected_buildings",
               facet_col="bairro", facet_col_wrap=3, category_orders={"bairro": sorted(bairros_prob)})
for t in orig.data:
    t.update(line=dict(color="lightgray", dash="dash"), showlegend=False)
    fig.add_trace(t)
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

Quedas restantes: 0


## Dica 2 — `percentage` e `affected_population` devem ser coerentes com os outros campos

**Regra esperada:**
* `percentage` = `affected_buildings / total_buildings × 100`
* `affected_population` = `population × percentage / 100`

Qualquer diferença além do arredondamento normal (±0,5) é uma inconsistência.

In [6]:
# Valores esperados a partir dos outros campos (dados originais)
df["pct_esperado"] = df["affected_buildings"] / df["total_buildings"] * 100
df["pop_esperada"] = df["population"] * df["percentage"] / 100      # usa a percentage do próprio arquivo
df["erro_pct"] = df["percentage"] - df["pct_esperado"]
df["erro_pop"] = df["affected_population"] - df["pop_esperada"]

inc_pct = df[df["erro_pct"].abs() > 0.5]
inc_pop = df[df["erro_pop"].abs() > 0.5]
cem_falso = df[(df["percentage"] == 100) & (df["affected_buildings"] < df["total_buildings"])]

print(f"percentage incoerente:          {len(inc_pct)} linhas ({len(inc_pct)/len(df):.0%}) | sempre para cima: {(inc_pct['erro_pct'] > 0).all()}")
print(f"affected_population incoerente: {len(inc_pop)} linhas ({len(inc_pop)/len(df):.0%}) | sempre para cima: {(inc_pop['erro_pop'] > 0).all()}")
print(f"100% sem todos os prédios afetados: {len(cem_falso)} linhas em {sorted(cem_falso['bairro'].unique())}")

# As duas colunas são arredondadas sempre para cima (ceil)?
print("percentage == ceil(pct_esperado):          ", (np.ceil(df["pct_esperado"]) == df["percentage"]).mean() == 1)
print("affected_population == ceil(pop_esperada): ", (np.ceil(df["pop_esperada"] - 1e-9) == df["affected_population"]).mean() == 1)

cols = ["bairro", "nivel_m", "affected_buildings", "total_buildings", "pct_esperado", "percentage",
        "population", "pop_esperada", "affected_population", "erro_pct"]
inc_pct.sort_values("erro_pct", ascending=False)[cols].head(10).round(2)

percentage incoerente:          1294 linhas (19%) | sempre para cima: True
affected_population incoerente: 1314 linhas (20%) | sempre para cima: True
100% sem todos os prédios afetados: 128 linhas em ['Arquipélago', 'Cidade Baixa', 'Farrapos', 'Floresta', 'Praia de Belas', 'Santa Maria Goretti', 'São Geraldo']
percentage == ceil(pct_esperado):           True
affected_population == ceil(pop_esperada):  True


,bairro,nivel_m,affected_buildings,total_buildings,pct_esperado,percentage,population,pop_esperada,affected_population,erro_pct
753,Boa Vista do Sul,7.3,686,2450,28.00,29,2703,783.87,784,1.00
752,Boa Vista do Sul,7.2,686,2450,28.00,29,2703,783.87,784,1.00
13,Aberta dos Morros,4.3,94,4696,2.00,3,9854,295.62,296,1.00
1337,Chapéu do Sol,8.9,935,1798,52.00,53,5547,2939.91,2940,1.00
6229,Vila Assunção,8.2,679,3086,22.00,23,3974,914.02,915,1.00
2074,Floresta,4.5,1791,3731,48.00,49,8798,4311.02,4312,1.00
2057,Farroupilha,9.9,84,227,37.00,38,774,294.12,295,1.00
2056,Farroupilha,9.8,84,227,37.00,38,774,294.12,295,1.00
4529,Pedra Redonda,8.6,152,447,34.00,35,570,199.50,200,1.00
1466,Cidade Baixa,7.6,2152,2339,92.01,93,13014,12103.02,12104,0.99


In [7]:
# Tratamento: recalcular a partir de affected_buildings (já corrigido na dica 1)
dc["percentage_inct"] = df["percentage"]
dc["affected_population_inct"] = df["affected_population"]

dc["percentage"] = (dc["affected_buildings"] / dc["total_buildings"] * 100).round(2)
dc["affected_population"] = (dc["population"] * dc["percentage"] / 100).round().astype(int)

dc["tratamento_d2"] = np.where(
    (dc["percentage"] - dc["percentage_inct"]).abs() > 0.5, "recalculado (percentage)",
    np.where(dc["affected_population"] != dc["affected_population_inct"], "recalculado (arredondamento)", ""))

print(dc["tratamento_d2"].value_counts())


tratamento_d2
                                4032
recalculado (arredondamento)    1351
recalculado (percentage)        1291
Name: count, dtype: int64


### Verificação: os campos derivados devem bater com os outros campos

In [8]:
pct_esperado = (dc["affected_buildings"] / dc["total_buildings"] * 100).round(2)
pop_esperada = dc["population"] * dc["percentage"] / 100
print("percentage incoerente:         ", int(((dc["percentage"] - pct_esperado).abs() > 1e-9).sum()))
print("affected_population incoerente:", int(((dc["affected_population"] - pop_esperada).abs() > 0.5).sum()))
print("100% sem todos afetados:       ", int(((dc["percentage"] == 100) & (dc["affected_buildings"] < dc["total_buildings"])).sum()))

# Total de pessoas afetadas por nível: valores do INCT x recalculados
tot = (dc.groupby("nivel_m")[["affected_population_inct", "affected_population"]].sum()
         .rename(columns={"affected_population_inct": "INCT (original)",
                          "affected_population": "Recalculado"})
         .reset_index().melt(id_vars="nivel_m", var_name="série", value_name="pessoas afetadas"))
fig = px.line(tot, x="nivel_m", y="pessoas afetadas", color="série", markers=True,
              color_discrete_map={"Recalculado": "#1f77b4", "INCT (original)": "lightgray"},
              labels={"nivel_m": "Nível do Guaíba (m)"},
              title="População afetada em Porto Alegre por nível")
fig.update_traces(selector=dict(name="INCT (original)"), line=dict(dash="dash"))
fig.show()

percentage incoerente:          0
affected_population incoerente: 0
100% sem todos afetados:        0


In [9]:
# Todos os níveis têm informação de todos os bairros?
todos_bairros = set(df["bairro"])
todos_niveis = sorted(df["nivel_m"].unique())
print(f"{len(todos_niveis)} níveis ({todos_niveis[0]} a {todos_niveis[-1]} m) e {len(todos_bairros)} bairros distintos")

# Combinações nível x bairro que deveriam existir e não existem
grade = pd.MultiIndex.from_product([todos_niveis, sorted(todos_bairros)], names=["nivel_m", "bairro"])
presentes = pd.MultiIndex.from_frame(df[["nivel_m", "bairro"]])
faltando = grade.difference(presentes).to_frame(index=False)

# Duplicatas (mesmo bairro duas vezes no mesmo nível)
duplicadas = df[df.duplicated(["nivel_m", "bairro"], keep=False)]

# Níveis faltando no intervalo (passo esperado de 0,1 m)
esperados = np.round(np.arange(todos_niveis[0], todos_niveis[-1] + 0.05, 0.1), 1)
niveis_faltando = sorted(set(esperados) - set(np.round(todos_niveis, 1)))

# Campos vazios
nulos = df[["affected_buildings", "total_buildings", "percentage", "population", "affected_population"]].isna().sum()

print("Bairros por nível:", df.groupby("nivel_m")["bairro"].nunique().value_counts().to_dict())
print("Combinações nível x bairro faltando:", len(faltando))
print("Linhas duplicadas:", len(duplicadas))
print("Níveis faltando no intervalo:", niveis_faltando or "nenhum")
print("Valores nulos:", int(nulos.sum()))

if len(faltando):
    display(faltando.groupby("nivel_m")["bairro"].apply(list).to_frame("bairros ausentes"))

71 níveis (3.0 a 10.0 m) e 94 bairros distintos
Bairros por nível: {94: 71}
Combinações nível x bairro faltando: 0
Linhas duplicadas: 0
Níveis faltando no intervalo: nenhum
Valores nulos: 0


# Limpeza dos dados

Reúne em um só lugar as verificações e os tratamentos encontrados acima e gera o JSON limpo em formato longo
(uma linha por nível e bairro), em `dados/processados/inct/`. Cada alteração está comentada com **ALTERAÇÃO**.

| Problema encontrado | Onde | Tratamento |
|---|---|---|
| Ordem dos níveis/bairros no arquivo não é confiável | dica 3 | ordenar por bairro e nível antes de qualquer cálculo |
| `affected_buildings` diminui quando o rio sobe (zeros isolados, pico, patamar inicial, ruído) | dica 1 | interpolar pelo nível dentro do bairro (17 valores) |
| `percentage` arredondada sempre para cima (até 100% sem todos os prédios afetados) | dica 2 | recalcular = `affected_buildings / total_buildings × 100` (2 casas) |
| `affected_population` calculada sobre a `percentage` arredondada | dica 2 | recalcular = `population × percentage / 100` (inteiro) |
| Dados faltando, duplicatas, níveis ou bairros ausentes | verificação abaixo | conferido: nenhum caso, nada a remover |

## Verificação de dados faltando e duplicatas (dados originais)

In [10]:
campos = ["affected_buildings", "total_buildings", "percentage", "population", "affected_population"]

# Nomes de bairro com espaços extras ou que só diferem em maiúsculas/minúsculas (formatos inconsistentes)
nomes = pd.Series(df["bairro"].unique())
variantes = nomes.groupby(nomes.str.strip().str.lower()).size()

verificacao = pd.DataFrame({
    "Verificação": [
        "Linhas (esperado 71 níveis × 94 bairros = 6674)",
        "Valores nulos",
        "Linhas duplicadas (mesmo nível e bairro)",
        "Combinações nível × bairro faltando",
        "Níveis faltando entre 3,0 e 10,0 m",
        "Bairros com nome em mais de uma grafia",
        "Valores negativos",
        "affected_buildings > total_buildings",
    ],
    "Resultado": [
        len(df),
        int(df[campos].isna().sum().sum()),
        int(df.duplicated(["nivel_m", "bairro"]).sum()),
        len(faltando),
        len(niveis_faltando),
        int((variantes > 1).sum()),
        int((df[campos] < 0).sum().sum()),
        int((df["affected_buildings"] > df["total_buildings"]).sum()),
    ],
})
verificacao

,Verificação,Resultado
0,Linhas (esperado 71 níveis × 94 bairros = 6674),6674
1,Valores nulos,0
2,Linhas duplicadas (mesmo nível e bairro),0
3,Combinações nível × bairro faltando,0
4,"Níveis faltando entre 3,0 e 10,0 m",0
5,Bairros com nome em mais de uma grafia,0
6,Valores negativos,0
7,affected_buildings > total_buildings,0


Nenhum dado faltando nem duplicado: não há linhas a remover ou preencher. Os problemas do arquivo são de
**coerência entre níveis e entre campos**, tratados nas dicas 1 e 2.

## Montagem da base limpa

In [11]:
# Os tratamentos das dicas 1 e 2 já foram aplicados em `dc` nas células acima:
#   ALTERAÇÃO (dica 3): base ordenada por bairro e nível
#   ALTERAÇÃO (dica 1): affected_buildings interpolado onde havia queda (coluna tratamento_d1)
#   ALTERAÇÃO (dica 2): percentage e affected_population recalculados (coluna tratamento_d2)
inct_limpo = dc[["nivel_m", "bairro"] + campos].copy()

# ALTERAÇÃO: coluna única que registra o que foi alterado em cada linha (vazio = sem alteração)
inct_limpo["tratamento"] = (dc["tratamento_d1"] + "; " + dc["tratamento_d2"]).str.strip("; ")

# ALTERAÇÃO: ordem final por nível e bairro (mesma organização do JSON original)
inct_limpo = inct_limpo.sort_values(["nivel_m", "bairro"]).reset_index(drop=True)
inct_limpo["nivel_m"] = inct_limpo["nivel_m"].round(1)

print(inct_limpo["tratamento"].replace("", "(sem alteração)").value_counts())
inct_limpo.head()

tratamento
(sem alteração)                                                          4032
recalculado (arredondamento)                                             1344
recalculado (percentage)                                                 1281
interpolado (ruído); recalculado (arredondamento)                           7
interpolado (patamar inicial inconsistente); recalculado (percentage)       5
interpolado (zero isolado); recalculado (percentage)                        4
interpolado (pico isolado); recalculado (percentage)                        1
Name: count, dtype: int64


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population,tratamento
0,3.0,Aberta dos Morros,0,4696,0.00,9854,0,
1,3.0,Agronomia,0,1771,0.00,2677,0,
2,3.0,Anchieta,28,589,4.75,791,38,recalculado (arredondamento)
3,3.0,Arquipélago,4680,4688,99.83,6411,6400,interpolado (ruído); recalculado (arredondamento)
4,3.0,Auxiliadora,0,1995,0.00,8909,0,


## Verificação da base limpa

In [12]:
assert inct_limpo.shape[0] == 6674                                    # 71 níveis × 94 bairros
assert inct_limpo[campos].notna().all().all()                          # sem nulos
assert not inct_limpo.duplicated(["nivel_m", "bairro"]).any()          # sem duplicatas
assert (inct_limpo.groupby("bairro")["affected_buildings"].diff().dropna() >= 0).all()   # dica 1: sem quedas
assert (inct_limpo["affected_buildings"] <= inct_limpo["total_buildings"]).all()
pct = (inct_limpo["affected_buildings"] / inct_limpo["total_buildings"] * 100).round(2)
assert (inct_limpo["percentage"] == pct).all()                         # dica 2: percentage coerente
pop = inct_limpo["population"] * inct_limpo["percentage"] / 100
assert ((inct_limpo["affected_population"] - pop).abs() <= 0.5).all()  # dica 2: população coerente
# total_buildings e population não foram alterados
orig = df.set_index(["nivel_m", "bairro"]).sort_index()
limpo = inct_limpo.set_index(["nivel_m", "bairro"]).sort_index()
assert (orig[["total_buildings", "population"]].values == limpo[["total_buildings", "population"]].values).all()

print("Todas as verificações passaram:", inct_limpo.shape)

Todas as verificações passaram: (6674, 8)


## Exportação

Formato longo: lista de objetos JSON, uma linha por nível e bairro, com os campos originais tratados e a
coluna `tratamento` indicando o que mudou (vazia quando a linha não foi alterada).

In [13]:
SAIDA = RAIZ / "dados" / "processados" / "inct"
SAIDA.mkdir(parents=True, exist_ok=True)
arquivo = SAIDA / "inct_limpo.json"

with open(arquivo, "w", encoding="utf-8") as f:
    json.dump(inct_limpo.to_dict(orient="records"), f, ensure_ascii=False, indent=1)

releitura = pd.read_json(arquivo)
print("Salvo:", arquivo, releitura.shape)
releitura.head()

Salvo: /home/piedrorn/Documentos/puc/coleta/projeto-enchentes/dados/processados/inct/inct_limpo.json (6674, 8)


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population,tratamento
0,3.0,Aberta dos Morros,0,4696,0.00,9854,0,
1,3.0,Agronomia,0,1771,0.00,2677,0,
2,3.0,Anchieta,28,589,4.75,791,38,recalculado (arredondamento)
3,3.0,Arquipélago,4680,4688,99.83,6411,6400,interpolado (ruído); recalculado (arredondamento)
4,3.0,Auxiliadora,0,1995,0.00,8909,0,
